Example Spark SQL

Luis Roberto Navarro Quinn

In [1]:
import findspark
findspark.init()

In [2]:
from spark_utils import SparkUtils

su = SparkUtils("Ejemplo Spark SQL", "spark://spark-master:7077")
su._spark.sparkContext

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/17 03:37:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=spark://spark-master:7077 appName=Ejemplo Spark SQL>

In [3]:
data = [
    (1, "Alice", 29),
    (2, "Bob", 35),
    (3, "Charlie", 41)
]

df = su._spark.createDataFrame(data)
df.printSchema()

root
 |-- _1: long (nullable = true)
 |-- _2: string (nullable = true)
 |-- _3: long (nullable = true)



In [4]:
from pyspark.sql.types import StructField, StructType, IntegerType, StringType, FloatType, TimestampType

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("nombre", StringType(), True),
    StructField("edad", IntegerType(), True)
])

df = su._spark.createDataFrame(data, schema)
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- edad: integer (nullable = true)



In [5]:
from datetime import datetime

factory_data = [
    ("M001", datetime(2026, 1, 26, 8, 0, 0), 75.3),
    ("M002", datetime(2026, 1, 26, 8, 5, 0), 68.7),
    ("M001", datetime(2026, 1, 26, 8, 10, 0), 76.1),
    ("M003", datetime(2026, 1, 26, 8, 15, 0), 72.4),
    ("M002", datetime(2026, 1, 26, 8, 20, 0), 69.8),
    ("M001", datetime(2026, 1, 26, 8, 25, 0), 77.5),
    ("M003", datetime(2026, 1, 26, 8, 30, 0), 73.2),
    ("M002", datetime(2026, 1, 26, 8, 35, 0), 70.1),
    ("M001", datetime(2026, 1, 26, 8, 40, 0), 78.0),
    ("M003", datetime(2026, 1, 26, 8, 45, 0), 74.6),
]

factory_schema = StructType([
    StructField("machine", StringType(), True),
    StructField("date", TimestampType(), True),
    StructField("temperature", FloatType(), True)
])

factory_df = su._spark.createDataFrame(factory_data, factory_schema)
factory_df.printSchema()
factory_df.show()

root
 |-- machine: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- temperature: float (nullable = true)



+-------+-------------------+-----------+
|machine|               date|temperature|
+-------+-------------------+-----------+
|   M001|2026-01-26 08:00:00|       75.3|
|   M002|2026-01-26 08:05:00|       68.7|
|   M001|2026-01-26 08:10:00|       76.1|
|   M003|2026-01-26 08:15:00|       72.4|
|   M002|2026-01-26 08:20:00|       69.8|
|   M001|2026-01-26 08:25:00|       77.5|
|   M003|2026-01-26 08:30:00|       73.2|
|   M002|2026-01-26 08:35:00|       70.1|
|   M001|2026-01-26 08:40:00|       78.0|
|   M003|2026-01-26 08:45:00|       74.6|
+-------+-------------------+-----------+



In [6]:
from pyspark.sql.functions import col
filtered_df = factory_df.filter(col("temperature") > 75)
filtered_df.show()

+-------+-------------------+-----------+
|machine|               date|temperature|
+-------+-------------------+-----------+
|   M001|2026-01-26 08:00:00|       75.3|
|   M001|2026-01-26 08:10:00|       76.1|
|   M003|2026-01-26 08:15:00|       72.4|
|   M001|2026-01-26 08:25:00|       77.5|
|   M003|2026-01-26 08:30:00|       73.2|
|   M002|2026-01-26 08:35:00|       70.1|
|   M001|2026-01-26 08:40:00|       78.0|
|   M003|2026-01-26 08:45:00|       74.6|
+-------+-------------------+-----------+



In [7]:
record_count = factory_df.count()
print(f"Total records: {record_count}")

[Stage 4:=============================>                             (1 + 1) / 2]

Total records: 10


In [8]:
ordered_df = factory_df.orderBy(col("temperature"),
ascending=False)

ordered_df.show()

[Stage 7:=============================>                             (1 + 1) / 2]

+-------+-------------------+-----------+
|machine|               date|temperature|
+-------+-------------------+-----------+
|   M001|2026-01-26 08:40:00|       78.0|
|   M001|2026-01-26 08:25:00|       77.5|
|   M001|2026-01-26 08:10:00|       76.1|
|   M001|2026-01-26 08:00:00|       75.3|
|   M003|2026-01-26 08:45:00|       74.6|
|   M003|2026-01-26 08:30:00|       73.2|
|   M003|2026-01-26 08:15:00|       72.4|
|   M002|2026-01-26 08:35:00|       70.1|
|   M002|2026-01-26 08:20:00|       69.8|
|   M002|2026-01-26 08:05:00|       68.7|
+-------+-------------------+-----------+



In [9]:
grouped_df = factory_df.groupBy("machine").count()
grouped_df.show()

[Stage 8:=============================>                             (1 + 1) / 2]

+-------+-----+
|machine|count|
+-------+-----+
|   M002|    3|
|   M003|    3|
|   M001|    4|
+-------+-----+



In [11]:
from pyspark.sql.functions import avg, min, max
agg_df = factory_df.groupBy(col("machine")).agg(
    avg("temperature").alias("avg_temp"),
    min("temperature").alias("min_temp"),
    max("temperature").alias("max_temp")
)

agg_df.show()

[Stage 14:=============================>                            (1 + 1) / 2]

+-------+-----------------+--------+--------+
|machine|         avg_temp|min_temp|max_temp|
+-------+-----------------+--------+--------+
|   M002|69.53333282470703|    68.7|    70.1|
|   M003|73.39999898274739|    72.4|    74.6|
|   M001|76.72500038146973|    75.3|    78.0|
+-------+-----------------+--------+--------+



In [13]:
ordered_df = factory_df.orderBy(col("temperature"),
ascending=False).limit(1)

ordered_df.show()

+-------+-------------------+-----------+
|machine|               date|temperature|
+-------+-------------------+-----------+
|   M001|2026-01-26 08:40:00|       78.0|
+-------+-------------------+-----------+

